# Assignment 4 — Classification: Breast Cancer Wisconsin

**Selected task:** binary classification — predict whether a breast mass is malignant or benign from numeric diagnostic features.

**Dataset:** Breast Cancer Wisconsin (Diagnostic), available through `scikit-learn`'s built-in dataset loader. It contains 569 observations, 30 numeric features and a binary target. This choice makes the notebook reproducible without downloading an additional dataset.

> If the instructor requires one of the exact Kaggle datasets listed in the assignment, replace the dataset-loading cell with that dataset. The analysis and modeling pipeline below are otherwise reusable.

## 1. Imports and reproducibility

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, GridSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)
from scipy.stats import wilcoxon

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load and describe the dataset

Target labels in the original sklearn dataset are `0 = malignant` and `1 = benign`.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
df = dataset.frame.copy()
df["diagnosis"] = df["target"].map({0: "malignant", 1: "benign"})
df = df.drop(columns="target")

TARGET = "diagnosis"
X = df.drop(columns=TARGET)
y = (df[TARGET] == "malignant").astype(int)  # positive class: malignant

print("Dataset shape:", df.shape)
print("Features:", X.shape[1])
print("Target classes:", y.value_counts().to_dict())
display(df.head())
display(df.describe().T)

## 3. Exploratory Data Analysis (EDA)

Research questions:
1. Are the classes balanced, or is accuracy alone potentially misleading?
2. Which numeric features show the strongest association with the diagnosis?
3. Can the two diagnosis classes be separated in a low-dimensional projection?

In [ ]:
# Missing values, duplicate rows and class balance
missing = df.isna().sum().sort_values(ascending=False)
print("Missing values (top):")
display(missing.head(10))
print("Duplicate rows:", df.duplicated().sum())
print("Class counts:")
display(df[TARGET].value_counts())

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

# Visualization 1: target distribution
df[TARGET].value_counts().plot(kind="bar", ax=axes[0, 0], rot=0)
axes[0, 0].set_title("1. Diagnosis class distribution")
axes[0, 0].set_ylabel("Count")

# Visualization 2: missing values
missing_positive = missing[missing > 0]
axes[0, 1].set_title("2. Missing values by feature")
axes[0, 1].set_ylabel("Missing count")
if len(missing_positive):
    missing_positive.plot(kind="bar", ax=axes[0, 1])
else:
    axes[0, 1].text(0.5, 0.5, "No missing values", ha="center", va="center",
                    transform=axes[0, 1].transAxes)
    axes[0, 1].set_xticks([])

# Visualization 3: selected feature distributions by class
for label, group in df.groupby(TARGET):
    axes[0, 2].hist(group["mean radius"], bins=20, alpha=0.55, label=label)
axes[0, 2].set_title("3. Mean radius by diagnosis")
axes[0, 2].set_xlabel("Mean radius")
axes[0, 2].legend()

# Visualization 4: correlation heatmap for the 10 most target-associated features
corr = X.assign(target=y).corr(numeric_only=True)["target"].drop("target")
top_features = corr.abs().sort_values(ascending=False).head(10).index
corr_subset = X[top_features].corr()
im = axes[1, 0].imshow(corr_subset, cmap="coolwarm", vmin=-1, vmax=1)
axes[1, 0].set_title("4. Correlation: top 10 features")
axes[1, 0].set_xticks(range(len(top_features)))
axes[1, 0].set_yticks(range(len(top_features)))
axes[1, 0].set_xticklabels(top_features, rotation=90, fontsize=6)
axes[1, 0].set_yticklabels(top_features, fontsize=6)
fig.colorbar(im, ax=axes[1, 0], fraction=0.046)

# Visualization 5: boxplot
df.boxplot(column="mean texture", by=TARGET, ax=axes[1, 1])
axes[1, 1].set_title("5. Mean texture by diagnosis")
axes[1, 1].set_xlabel("Diagnosis")
axes[1, 1].set_ylabel("Mean texture")
fig.suptitle("")
plt.tight_layout()
plt.show()

# Visualization 6: PCA projection (standardized features)
X_scaled = StandardScaler().fit_transform(X)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
plt.figure(figsize=(8, 6))
for label, color in [(0, "tab:red"), (1, "tab:blue")]:
    mask = y.to_numpy() == label
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.65,
                label="malignant" if label == 0 else "benign", s=25)
plt.title("6. PCA projection of the observations")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.legend()
plt.tight_layout()
plt.show()

print("Potential outliers: inspect numeric feature boxplots; extreme values may be valid measurements.")
print("Top absolute correlations with malignant label:")
display(corr.abs().sort_values(ascending=False).head(10).to_frame("absolute_correlation"))

### EDA observations and preprocessing decisions

- The dataset has numeric measurements and a binary target.
- The positive class is defined as **malignant** to make recall for malignant cases explicit.
- Missing values are checked and handled with median imputation inside each model pipeline.
- Duplicate rows are checked; no rows are removed automatically because repeated observations should be reviewed before deletion.
- Standardization is used for Logistic Regression and SVM, which are sensitive to feature scale.
- The train/test split is stratified to preserve class proportions.

## 4. Machine Learning

Formal problem: given a feature vector of 30 diagnostic measurements, estimate the binary label `malignant` (1) or `benign` (0).

Three different algorithms are compared:
- Logistic Regression — linear, interpretable baseline.
- Random Forest — nonlinear ensemble of decision trees.
- Support Vector Machine (RBF) — nonlinear margin-based classifier.

A stratified 80/20 split is used. Hyperparameter optimization is performed for Random Forest using 5-fold cross-validation and F1 score.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=3000, class_weight="balanced",
                                     random_state=RANDOM_STATE))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(class_weight="balanced",
                                         random_state=RANDOM_STATE,
                                         n_jobs=-1))
    ]),
    "SVM (RBF)": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(probability=True, class_weight="balanced",
                      random_state=RANDOM_STATE))
    ])
}

scoring = {"accuracy": "accuracy", "precision": "precision",
           "recall": "recall", "f1": "f1", "roc_auc": "roc_auc"}

cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring,
                            n_jobs=-1, return_train_score=False)
    cv_rows.append({
        "Model": name,
        **{metric: scores[f"test_{metric}"].mean() for metric in scoring},
        "F1 std": scores["test_f1"].std()
    })

cv_results = pd.DataFrame(cv_rows).sort_values("f1", ascending=False)
display(cv_results)

### 4.1 Hyperparameter optimization — Random Forest

In [ ]:
rf_pipeline = models["Random Forest"]
param_grid = {
    "model__n_estimators": [150, 300],
    "model__max_depth": [None, 8, 15],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", 0.7]
}

grid = GridSearchCV(
    rf_pipeline, param_grid, scoring="f1", cv=cv, n_jobs=-1, refit=True
)
grid.fit(X_train, y_train)
best_rf = grid.best_estimator_

print("Best Random Forest parameters:")
print(grid.best_params_)
print("Best mean CV F1:", round(grid.best_score_, 4))

### 4.2 Test-set comparison

The test set is kept separate from cross-validation and used only for final evaluation. Metrics include accuracy, precision, recall, F1 and ROC-AUC.

In [ ]:
final_models = {
    "Logistic Regression": models["Logistic Regression"],
    "Tuned Random Forest": best_rf,
    "SVM (RBF)": models["SVM (RBF)"]
}

test_rows = []
for name, model in final_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

test_results = pd.DataFrame(test_rows).sort_values("F1", ascending=False)
display(test_results)

ax = test_results.set_index("Model")[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]].plot(
    kind="bar", figsize=(12, 6), ylim=(0, 1), rot=15
)
ax.set_title("Model performance on held-out test set")
ax.set_ylabel("Score")
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix and ROC curves for the best test-set model
best_name = test_results.iloc[0]["Model"]
best_model = final_models[best_name]
best_pred = best_model.predict(X_test)
best_prob = best_model.predict_proba(X_test)[:, 1]

ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred, display_labels=["benign", "malignant"],
    cmap="Blues", values_format="d"
)
plt.title(f"Confusion matrix — {best_name}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
for name, model in final_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=plt.gca())
plt.plot([0, 1], [0, 1], "k--", label="Random baseline")
plt.title("ROC curves")
plt.tight_layout()
plt.show()

print("Classification report for selected model:")
print(classification_report(y_test, best_pred,
                            target_names=["benign", "malignant"],
                            zero_division=0))

### Model comparison: strengths and limitations

- **Logistic Regression:** fast and interpretable baseline; may underfit nonlinear relationships.
- **Random Forest:** captures nonlinear interactions and is robust to feature scaling; can be less transparent and more computationally expensive.
- **SVM (RBF):** can model complex boundaries; requires scaling and may be slower on larger datasets.

For a medical screening context, malignant-class recall is especially important because false negatives can be costly. The best model should therefore be selected using the metric priorities of the application, not accuracy alone.

## 5. Hypothesis testing

Compare Logistic Regression and the tuned Random Forest using paired F1 scores from the same stratified folds.

- $H_0$: the two algorithms have the same performance distribution (no systematic difference in F1).
- $H_1$: their F1 performance differs.

A two-sided Wilcoxon signed-rank test is used on paired fold scores. The significance level is 0.05.

In [ ]:
# Use identical folds for both algorithms to obtain paired scores.
logistic_for_test = models["Logistic Regression"]
rf_for_test = best_rf

logistic_scores = cross_validate(
    logistic_for_test, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1
)["test_score"]
rf_scores = cross_validate(
    rf_for_test, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1
)["test_score"]

print("Logistic Regression fold F1:", np.round(logistic_scores, 4))
print("Tuned Random Forest fold F1:", np.round(rf_scores, 4))

# Wilcoxon requires at least one non-zero paired difference.
differences = logistic_scores - rf_scores
if np.allclose(differences, 0):
    statistic, p_value = 0.0, 1.0
else:
    statistic, p_value = wilcoxon(logistic_scores, rf_scores,
                                  alternative="two-sided",
                                  zero_method="wilcox")

alpha = 0.05
print(f"Wilcoxon statistic: {statistic:.4f}")
print(f"p-value: {p_value:.6f}")
if p_value < alpha:
    print("Reject H0: statistically significant difference in paired F1 scores.")
else:
    print("Fail to reject H0: insufficient evidence of a statistically significant difference.")

## 6. Final conclusion

This notebook explores the Breast Cancer Wisconsin dataset, compares three classification algorithms, tunes Random Forest, evaluates the final models on a held-out test set and tests the difference between Logistic Regression and Random Forest using paired cross-validation F1 scores.

The selected model and its metrics are displayed by the notebook at runtime. In this classification task, recall for the malignant class is an important consideration alongside F1 and ROC-AUC.